In [10]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

In [11]:
repo_dir = next(p for p in Path.cwd().parents if (p / '.git').exists())
trial_results_dir = repo_dir / 'trial_results'

def parse_csv_filename(csv_filename):
    """Extract world name and planner type from CSV filename"""
    # Parse world SDF file path and planner name using regex
    # Following the naming schema of "world_<world_number>_<planner>_YYYYMMDD_HHMMSS.csv"
    csv_name_pattern = r'(world_\d+)_(.+)_(\d{8})_(\d{6}).csv$'
    match = re.match(csv_name_pattern, csv_filename)

    if match:
        world_name      = match.group(1)    # world_<world_number>
        planner_type    = match.group(2)    # APF or TD3_00380_1000   
        date_str        = match.group(3)    # YYYYMMDD
        time_str        = match.group(4)    # HHMMSS
    else:
        raise ValueError(f"CSV filename {csv_filename} does not match expected pattern")

    return world_name, planner_type

# print(f"{'Planner':^15} | {'SPL':^7} | {'Success Rate (%)'}")
# print('-'*45)

csv_paths = sorted(trial_results_dir.glob("*.csv"))
planner_list = np.empty(len(csv_paths), dtype=object)
SPL_avg_list = np.zeros(len(csv_paths))
SPL_std_list = np.zeros(len(csv_paths))
success_rate_list = np.zeros(len(csv_paths))

for idx, csv_path in enumerate(csv_paths):
    df = pd.read_csv(csv_path)

    world_name, planner_type = parse_csv_filename(csv_path.name)

    success = df['success']
    straight_line_distance = df['straight_line_distance']
    distance_traveled_odom = df['distance_traveled_odom']

    # Success rates
    success_rate = np.mean(success) * 100
    
    # Calculate Success weighted by Path Length (SPL)
    spl = success * (straight_line_distance / distance_traveled_odom)
    spl_avg = np.mean(spl)
    spl_stdev = np.std(spl)

    planner_list[idx] = planner_type
    SPL_avg_list[idx] = np.round(spl_avg, 5)
    SPL_std_list[idx] = np.round(spl_stdev, 5)
    success_rate_list[idx] = np.round(success_rate, 5)

    # print(f"{planner_type:15}   {spl_avg: 5.3f}   {success_rate: 11.2f}")

result_df = pd.DataFrame({
    "SPL avg": SPL_avg_list,
    "SPL std": SPL_std_list, 
    "Success Rate (%)": success_rate_list
}, index=planner_list)

# result_df.set_index("Planner")
result_df

,SPL avg,SPL std,Success Rate (%)
APF,0.75810,0.25513,93.52518
TD3_00377_1000,0.89992,0.20832,95.68345
TD3_00377_900,0.83336,0.22126,94.96403
TD3_00378_1000,0.84275,0.28780,90.64748
TD3_00378_900,0.88108,0.19232,98.56115
TD3_00379_1000,0.83260,0.29353,89.92806
TD3_00379_800,0.86861,0.20090,97.12230
TD3_00380_1000,0.85547,0.25704,92.80576
TD3_00380_900,0.86582,0.24861,94.24460
TD3_00381_1000,0.83107,0.25392,94.96403


In [12]:
# TD3_direct statistics
TD3_direct_runs = ["TD3_00377_1000", "TD3_00378_1000", "TD3_00379_1000", "TD3_00380_1000", "TD3_00381_1000"]
TD3_direct_SPL = result_df.loc[TD3_direct_runs, 'SPL avg']
TD3_direct_success = result_df.loc[TD3_direct_runs, 'Success Rate (%)']
print(rf"TD3_direct SPL: {np.mean(TD3_direct_SPL):.3f} ± {np.std(TD3_direct_SPL):.3f}")
print(rf"TD3_direct success: {np.mean(TD3_direct_success):.3f} ± {np.std(TD3_direct_success):.3f}")
print('-'*50)

# TD3_delta* statistics
TD3_direct_star_runs = ["TD3_00396_500", "TD3_00397_1000", "TD3_00398_1000", "TD3_00399_900", "TD3_00400_700",
                        "TD3_00401_900", "TD3_00402_700", "TD3_00403_900", "TD3_00404_500", "TD3_00405_1000"]
TD3_direct_star_SPL = result_df.loc[TD3_direct_star_runs, 'SPL avg']
TD3_direct_star_success = result_df.loc[TD3_direct_star_runs, 'Success Rate (%)']
print(rf"TD3_delta* SPL: {np.mean(TD3_direct_star_SPL):.3f} ± {np.std(TD3_direct_star_SPL):.3f}")
print(rf"TD3_delta* success: {np.mean(TD3_direct_star_success):.3f} ± {np.std(TD3_direct_star_success):.3f}")
print('-'*50)

# TD3_delta statistics
TD3_delta_runs = [f"TD3_{run_num:05d}_1000" for run_num in range(386,396)]
# TD3_delta_runs = ["TD3_00386_1000", "TD3_00387_1000", "TD3_00388_1000", "TD3_00389_1000", "TD3_00390_1000"]
TD3_delta_SPL = result_df.loc[TD3_delta_runs, 'SPL avg']
TD3_delta_success = result_df.loc[TD3_delta_runs, 'Success Rate (%)']
print(rf"TD3_delta SPL: {np.mean(TD3_delta_SPL):.3f} ± {np.std(TD3_delta_SPL):.3f}")
print(rf"TD3_delta success: {np.mean(TD3_delta_success):.3f} ± {np.std(TD3_delta_success):.3f}")
print('-'*50)

# TD3_delta* statistics
TD3_delta_star_runs = ["TD3_00386_1000", "TD3_00387_500", "TD3_00388_900", "TD3_00389_900", "TD3_00390_800",
                       "TD3_00391_800", "TD3_00392_600", "TD3_00393_900", "TD3_00394_800", "TD3_00395_900"]
TD3_delta_star_SPL = result_df.loc[TD3_delta_star_runs, 'SPL avg']
TD3_delta_star_success = result_df.loc[TD3_delta_star_runs, 'Success Rate (%)']
print(rf"TD3_delta* SPL: {np.mean(TD3_delta_star_SPL):.3f} ± {np.std(TD3_delta_star_SPL):.3f}")
print(rf"TD3_delta* success: {np.mean(TD3_delta_star_success):.3f} ± {np.std(TD3_delta_star_success):.3f}")
print('-'*50)

TD3_direct SPL: 0.852 ± 0.025
TD3_direct success: 92.806 ± 2.275
--------------------------------------------------
TD3_delta* SPL: 0.855 ± 0.031
TD3_delta* success: 94.892 ± 2.957
--------------------------------------------------
TD3_delta SPL: 0.718 ± 0.158
TD3_delta success: 83.597 ± 18.757
--------------------------------------------------
TD3_delta* SPL: 0.796 ± 0.057
TD3_delta* success: 92.302 ± 6.079
--------------------------------------------------
